# Phase 03 — Literature & Baseline Research
## CodeMentor-LLM
Testing meta-llama/Meta-Llama-3-8B-Instruct base model on 10 coding questions
before any fine-tuning to justify our project.

#  Libraries

In [3]:
!pip install -q transformers==4.49.0 bitsandbytes==0.45.3 accelerate==1.5.1 peft==0.14.0

# Checking GPU

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

CUDA available: True
GPU: Tesla T4
Memory: 14.56 GB


# Login to HuggingFace

In [5]:
from huggingface_hub import login
login()

# Load meta-llama/Meta-Llama-3-8B-Instruct in 4-bit

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded successfully
Memory footprint: 5.21 GB


# Create inference function

In [9]:
def generate_response(prompt, max_new_tokens=256):
    # Apply Mistral chat template
    messages = [{"role": "user", "content": prompt}]

    # Tokenize
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

print("Inference function ready")

Inference function ready


# Run 10 coding questions against base model

In [10]:
# 10 coding questions to test base model
questions = [
    "Write a Python function to reverse a string.",
    "Explain what a decorator is in Python with an example.",
    "What is the difference between a list and a tuple in Python?",
    "Write a Python function to check if a number is prime.",
    "How do you handle exceptions in Python? Show an example.",
    "Write a SQL query to find the second highest salary from a table.",
    "Explain the concept of recursion with a Python example.",
    "What is the difference between == and is in Python?",
    "Write a Python function to find duplicates in a list.",
    "Explain what a REST API is in simple terms.",
]

# Run all 10 questions
results = []
for i, question in enumerate(questions):
    print(f"\n{'='*30}")
    print(f"Q{i+1}: {question}")
    print(f"{'='*30}")
    response = generate_response(question)
    print(f"A: {response}")
    results.append({"question": question, "response": response})

print("\nAll 10 questions completed")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Q1: Write a Python function to reverse a string.
A: Here is a simple Python function to reverse a string:
```
def reverse_string(s):
    return s[::-1]
```
This function uses slicing to extract the characters of the input string in reverse order. The `::-1` slice means "start at the end of the string and move backwards to the beginning, stepping backwards by 1 character each time".

For example, if you call the function with the string `"hello"`, it will return the string `"olleh"`.

Alternatively, you can use the `reversed` function and the `join` method to achieve the same result:
```
def reverse_string(s):
    return "".join(reversed(s))
```
This function uses the `reversed` function to reverse the characters of the input string, and then joins them together into a new string using the `join` method.

Both of these functions will work correctly for any input string.

Q2: Explain what a decorator is in Python with an example.
A: In Python, a decorator is a small, reusable function t

## Base Model Analysis — meta-llama/Meta-Llama-3-8B-Instruct

### What the base model does WELL:
- Answers all 10 coding questions correctly
- Formats code blocks consistently
- Gives clear explanations with step-by-step breakdowns (Q4, Q7)
- Handles multiple question types — Python, SQL, concepts
- Uses bold headers to structure answers (Q3, Q8)
- Provides real-world analogies for concepts (Q10 — restaurant analogy)
- Better code quality than Mistral baseline

### What the base model FAILS at:
- Responses still cut off mid-sentence (Q2, Q3, Q7, Q8, Q9)
- No consistent response length
- Attention mask warning during inference
- No specific coding instruction format
- Sometimes over-explains simple concepts
- SQL answer (Q6) uses incorrect syntax for some databases

### Why Fine-Tuning is still justified:
1. Responses cut off frequently — needs max_length tuning
2. No consistent coding instruction format
3. SFT on CodeAlpaca-20K will teach structured complete responses
4. DPO will align model to prefer higher quality responses
5. Fine-tuned model will be domain-specific and more focused

# Save results to JSON

In [11]:
import json

# Save baseline results
baseline_results = {
    "model": "meta-llama/Meta-Llama-3-8B-Instruct",
    "type": "base_model_no_finetuning",
    "results": results,
    "observations": {
        "strengths": [
            "Answers basic coding questions correctly",
            "Formats code blocks properly",
            "Gives multiple approaches",
            "Explains concepts clearly"
        ],
        "weaknesses": [
            "Responses get cut off mid-sentence",
            "No consistent response structure",
            "No step-by-step teaching format",
            "Inconsistent response length",
            "Attention mask warning"
        ],
        "finetuning_justified": True
    }
}

# Save to file
with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline results saved to baseline_results.json")

Baseline results saved to baseline_results.json
